# CUDA + GPU Profiling mini-project (Colab)

**Before you start:** menu **Runtime > Change runtime type > Hardware accelerator: T4 GPU > Save.**

Then run each cell top to bottom. You need two numbers at the end:
1. the **speedup** your CUDA kernel gets over the CPU, and
2. the **top GPU kernel** from the PyTorch profiler.


## 0. Confirm a GPU is attached

In [ ]:
!nvidia-smi

## 1. Write and run a CUDA C/C++ kernel (tiled matrix multiply)
The next cell writes a real `.cu` file; the one after compiles it with `nvcc` and runs it.

In [ ]:
%%writefile matmul.cu
#include <cstdio>
#include <cstdlib>
#include <cmath>
#include <cuda_runtime.h>

#define TILE 16

// Tiled matrix multiply using shared memory: C = A * B  (N x N, float)
__global__ void matmulTiled(const float* A, const float* B, float* C, int N) {
    __shared__ float As[TILE][TILE];
    __shared__ float Bs[TILE][TILE];
    int row = blockIdx.y * TILE + threadIdx.y;
    int col = blockIdx.x * TILE + threadIdx.x;
    float sum = 0.0f;
    for (int t = 0; t < (N + TILE - 1) / TILE; ++t) {
        if (row < N && t*TILE + threadIdx.x < N)
            As[threadIdx.y][threadIdx.x] = A[row*N + t*TILE + threadIdx.x];
        else
            As[threadIdx.y][threadIdx.x] = 0.0f;
        if (col < N && t*TILE + threadIdx.y < N)
            Bs[threadIdx.y][threadIdx.x] = B[(t*TILE + threadIdx.y)*N + col];
        else
            Bs[threadIdx.y][threadIdx.x] = 0.0f;
        __syncthreads();
        for (int k = 0; k < TILE; ++k)
            sum += As[threadIdx.y][k] * Bs[k][threadIdx.x];
        __syncthreads();
    }
    if (row < N && col < N)
        C[row*N + col] = sum;
}

// Simple CPU reference (naive triple loop)
void matmulCPU(const float* A, const float* B, float* C, int N) {
    for (int i = 0; i < N; ++i)
        for (int j = 0; j < N; ++j) {
            float s = 0.0f;
            for (int k = 0; k < N; ++k) s += A[i*N+k] * B[k*N+j];
            C[i*N+j] = s;
        }
}

int main() {
    int N = 1024;
    size_t bytes = (size_t)N * N * sizeof(float);
    float *hA = (float*)malloc(bytes);
    float *hB = (float*)malloc(bytes);
    float *hC = (float*)malloc(bytes);
    float *hRef = (float*)malloc(bytes);
    for (int i = 0; i < N*N; ++i) { hA[i] = (float)(rand()%10)/10.0f; hB[i] = (float)(rand()%10)/10.0f; }

    float *dA, *dB, *dC;
    cudaMalloc(&dA, bytes); cudaMalloc(&dB, bytes); cudaMalloc(&dC, bytes);
    cudaMemcpy(dA, hA, bytes, cudaMemcpyHostToDevice);
    cudaMemcpy(dB, hB, bytes, cudaMemcpyHostToDevice);

    dim3 block(TILE, TILE);
    dim3 grid((N+TILE-1)/TILE, (N+TILE-1)/TILE);

    // warm-up
    matmulTiled<<<grid, block>>>(dA, dB, dC, N);
    cudaDeviceSynchronize();

    // Time the GPU kernel with CUDA events
    cudaEvent_t s, e; cudaEventCreate(&s); cudaEventCreate(&e);
    cudaEventRecord(s);
    matmulTiled<<<grid, block>>>(dA, dB, dC, N);
    cudaEventRecord(e);
    cudaEventSynchronize(e);
    float gpu_ms = 0.0f; cudaEventElapsedTime(&gpu_ms, s, e);
    cudaMemcpy(hC, dC, bytes, cudaMemcpyDeviceToHost);

    // Time the CPU version
    clock_t c0 = clock();
    matmulCPU(hA, hB, hRef, N);
    clock_t c1 = clock();
    double cpu_ms = 1000.0 * (double)(c1 - c0) / CLOCKS_PER_SEC;

    // Verify correctness
    float maxErr = 0.0f;
    for (int i = 0; i < N*N; ++i) maxErr = fmaxf(maxErr, fabsf(hC[i] - hRef[i]));

    printf("Matrix size        : %d x %d\n", N, N);
    printf("GPU (tiled) time   : %.3f ms\n", gpu_ms);
    printf("CPU (naive) time   : %.3f ms\n", cpu_ms);
    printf("Speedup (CPU/GPU)  : %.1fx\n", cpu_ms / gpu_ms);
    printf("Max abs error      : %.6f  (should be ~0)\n", maxErr);

    cudaFree(dA); cudaFree(dB); cudaFree(dC);
    free(hA); free(hB); free(hC); free(hRef);
    return 0;
}


In [ ]:
!nvcc -O2 matmul.cu -o matmul && ./matmul

## 2. Profile a PyTorch model on the GPU
Runs ResNet-18 on the GPU and prints the GPU kernels sorted by time. The top row is your profiling finding.

In [ ]:
import torch, torchvision
from torch.profiler import profile, ProfilerActivity

assert torch.cuda.is_available(), "No GPU! Runtime > Change runtime type > T4 GPU, then re-run."
dev = torch.device("cuda")
print("GPU:", torch.cuda.get_device_name(0))

model = torchvision.models.resnet18().to(dev).eval()
x = torch.randn(16, 3, 224, 224, device=dev)

# warm-up
for _ in range(3):
    with torch.no_grad():
        model(x)
torch.cuda.synchronize()

with profile(activities=[ProfilerActivity.CPU, ProfilerActivity.CUDA]) as prof:
    with torch.no_grad():
        for _ in range(10):
            model(x)
    torch.cuda.synchronize()

# Top GPU kernels by total CUDA time -> read the #1 line as your "finding"
print(prof.key_averages().table(sort_by="cuda_time_total", row_limit=10))


## 3. Write these down for the resume
- **Speedup:** the `Speedup (CPU/GPU)` line from cell 1 (and confirm `Max abs error` is ~0, i.e. the kernel is correct).
- **Profiling finding:** the name of the #1 GPU kernel in cell 2's table and roughly what share of time it took.

Then send both lines back to continue the resume update.